# 02. Train all models (v2)
Chooses the early-stopping epoch on the **validation** set, then trains every model for 5 seeds and saves each one with a metadata file (privacy accounting included).
Use the normal **CPU** runtime. Finished models are skipped, so you can rerun after a disconnect.

In [ ]:
import os, sys, json, gc, time, warnings, subprocess
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
ROOT = os.environ.get("PA_ROOT", "/content/drive/MyDrive/privacy-audit/")
sys.path.insert(0, ROOT)          # pa_lib.py must sit in the privacy-audit folder on Drive
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pip('opacus')
from pa_lib import *
make_dirs()
print(versions())


Mounted at /content/drive
{'python': '3.13.18', 'numpy': '2.1.3', 'pandas': '2.2.3', 'torch': '2.11.0+cpu', 'opacus': '1.6.0', 'art': None, 'sklearn': '1.6.1'}


In [ ]:
from sklearn.metrics import roc_auc_score
Xtr, ytr = load_split("target_train"); Xva, yva = load_split("target_val")
Xs_tr, ys_tr = load_split("shadow_train")
input_dim = Xtr.shape[1]
print("train rows:", len(Xtr), "| validation rows:", len(Xva), "| features:", input_dim)

rows = []
def make_cb(seed):
    def cb(m, ep):
        pv, ptr = proba(m, Xva)[:, 1], proba(m, Xtr)[:, 1]
        rows.append({"seed": seed, "epoch": ep, "train_acc": float(((ptr >= .5) == ytr).mean()),
                     "val_acc": float(((pv >= .5) == yva).mean()), "val_auc": float(roc_auc_score(yva, pv))})
    return cb
for s in (1, 2, 3):
    train_plain(Xtr, ytr, EPOCHS, s, on_epoch=make_cb(s))
curve = pd.DataFrame(rows).groupby("epoch").mean(numeric_only=True).drop(columns="seed")
curve.to_csv(RESULTS + "early_stop_validation_curve.csv")
print(curve.loc[curve["val_auc"].nlargest(5).index].round(4))
EARLY_OVERRIDE = None          # set a number here only if you want to force the epoch
EARLY = int(EARLY_OVERRIDE or curve["val_auc"].idxmax())
set_config(early_epochs=EARLY, full_epochs=EPOCHS, n_train=int(len(Xtr)))
print("early-stopping epoch chosen on validation data:", EARLY)


train rows: 5969 | validation rows: 1601 | features: 118
       train_acc  val_acc  val_auc
epoch                             
6         0.8883   0.8813   0.6728
4         0.8882   0.8813   0.6712
5         0.8883   0.8809   0.6704
7         0.8885   0.8815   0.6703
3         0.8879   0.8813   0.6673
early-stopping epoch chosen on validation data: 6


In [ ]:
jobs = []
for seed in SEEDS:
    jobs += [("baseline_e50", seed, "plain", EPOCHS), ("baseline_early", seed, "plain", EARLY)]
    jobs += [(f"dpsgd_eps{e}", seed, "dp", e) for e in EPSILONS]
t0 = time.time()
for name, seed, kind, arg in jobs:
    if os.path.exists(f"{FINAL}{name}_seed{seed}.pt"):
        continue
    if kind == "plain":
        m, meta = train_plain(Xtr, ytr, arg, seed)
    else:
        m, meta = train_dp(Xtr, ytr, arg, seed)
    save_model(m, name, seed, meta)
    del m; gc.collect()
    print(f"{name} seed {seed} done ({(time.time() - t0) / 60:.1f} min)")

# the attacker's practice model used by the dashboard
if not os.path.exists(f"{FINAL}shadow_e50_seed1.pt"):
    m, meta = train_plain(Xs_tr, ys_tr, EPOCHS, 1001)
    save_model(m, "shadow_e50", 1, meta)
print("all models trained")


baseline_e50 seed 1 done (0.2 min)
baseline_early seed 1 done (0.3 min)
dpsgd_eps0.5 seed 1 done (0.6 min)
dpsgd_eps1 seed 1 done (0.9 min)
dpsgd_eps2 seed 1 done (1.3 min)
dpsgd_eps5 seed 1 done (1.7 min)
dpsgd_eps10 seed 1 done (2.2 min)
dpsgd_eps100 seed 1 done (3.9 min)
baseline_e50 seed 2 done (4.0 min)
baseline_early seed 2 done (4.0 min)
dpsgd_eps0.5 seed 2 done (4.4 min)
dpsgd_eps1 seed 2 done (4.7 min)
dpsgd_eps2 seed 2 done (5.1 min)
dpsgd_eps5 seed 2 done (5.5 min)
dpsgd_eps10 seed 2 done (5.9 min)
dpsgd_eps100 seed 2 done (7.7 min)
baseline_e50 seed 3 done (7.8 min)
baseline_early seed 3 done (7.8 min)
dpsgd_eps0.5 seed 3 done (8.1 min)
dpsgd_eps1 seed 3 done (8.5 min)
dpsgd_eps2 seed 3 done (8.8 min)
dpsgd_eps5 seed 3 done (9.3 min)
dpsgd_eps10 seed 3 done (9.7 min)
dpsgd_eps100 seed 3 done (11.6 min)
baseline_e50 seed 4 done (11.7 min)
baseline_early seed 4 done (11.7 min)
dpsgd_eps0.5 seed 4 done (12.0 min)
dpsgd_eps1 seed 4 done (12.4 min)
dpsgd_eps2 seed 4 done (12.8 m

In [ ]:
# privacy accounting for every DP model
acc = []
for e in EPSILONS:
    for s in SEEDS:
        acc.append(json.load(open(f"{FINAL}dpsgd_eps{e}_seed{s}.json")))
acc = pd.DataFrame(acc)
cols = ["target_epsilon", "spent_epsilon", "delta", "noise_multiplier", "max_grad_norm", "sample_rate", "steps", "epochs", "n_train"]
table = acc.groupby("target_epsilon")[cols[1:]].mean().round(6)
table.to_csv(RESULTS + "dp_accounting.csv")
print(table.to_string())
print("\nprivacy unit:", acc["privacy_unit"].iloc[0], "| accountant:", acc["accountant"].iloc[0], "| sampling:", acc["sampling"].iloc[0])
print("delta", DELTA, "should be smaller than 1 / training rows =", round(1 / len(Xtr), 6))


                spent_epsilon    delta  noise_multiplier  max_grad_norm  sample_rate   steps  epochs  n_train
target_epsilon                                                                                               
0.5                  0.496823  0.00001          7.500000            1.0     0.021277  2350.0    50.0   5969.0
1.0                  0.998822  0.00001          3.984375            1.0     0.021277  2350.0    50.0   5969.0
2.0                  1.995010  0.00001          2.216797            1.0     0.021277  2350.0    50.0   5969.0
5.0                  4.995411  0.00001          1.164551            1.0     0.021277  2350.0    50.0   5969.0
10.0                 9.991611  0.00001          0.823669            1.0     0.021277  2350.0    50.0   5969.0
100.0               99.997894  0.00001          0.380945            1.0     0.021277  2350.0    50.0   5969.0

privacy unit: single hospital visit (record level) | accountant: rdp | sampling: poisson
delta 1e-05 should be smaller 